# 🎨 LoRA Training — Alice (SDXL) в Google Colab**Тренировка LoRA на 15 фото Алисы**Использует [ai-toolkit](https://github.com/ostris/ai-toolkit) от Ostris — самый современный фреймворк для LoRA.**GPU:** T4 (бесплатно) или A100 (Pro)**Время:** ~30-60 минут**Результат:** `.safetensors` файл LoRA

## Шаг 1: Подключение Google Drive и настройка

In [ ]:
#@title ⚙️ Настройка и установкаfrom google.colab import drivedrive.mount('/content/drive')# ПутиGDRIVE_BASE = '/content/drive/MyDrive/06_PROJEKTE/Alice_LoRA'DATASET_DIR = f'{GDRIVE_BASE}/dataset'OUTPUT_DIR = f'{GDRIVE_BASE}/output'import osos.makedirs(OUTPUT_DIR, exist_ok=True)# Проверяем датасетdataset_files = os.listdir(DATASET_DIR) if os.path.exists(DATASET_DIR) else []pngs = [f for f in dataset_files if f.endswith('.png')]txts = [f for f in dataset_files if f.endswith('.txt')]print(f'📁 Датасет: {len(pngs)} изображений, {len(txts)} текстовых файлов')if pngs:    print(f'   Примеры: {pngs[:3]}')# Устанавливаем ai-toolkitprint('\n⏳ Установка ai-toolkit...')!git clone https://github.com/ostris/ai-toolkit.git /content/ai-toolkit%cd /content/ai-toolkit!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121!pip install -q -r requirements.txt!pip install -q xformersprint('\n✅ Установка завершена!')

## Шаг 2: Конфигурация тренировки

In [ ]:
#@title 🎯 Параметры тренировки#@markdown ### Основные параметрыTRIGGER_WORD = "alice"  #@param {type:"string"}MODEL_NAME = "stabilityai/stable-diffusion-xl-base-1.0"  #@param {type:"string"}RESOLUTION = 1024  #@param [512, 768, 1024] {type:"raw"}NUM_EPOCHS = 15  #@param {type:"slider", min:5, max:30, step:1}BATCH_SIZE = 1  #@param {type:"slider", min:1, max:4, step:1}LEARNING_RATE = 0.0001  #@param {type:"number"}RANK = 16  #@param [8, 16, 32, 64] {type:"raw"}ALPHA = 16  #@param [8, 16, 32, 64] {type:"raw"}# Генерируем конфигconfig = f'''{  "model": {    "name_or_path": "{MODEL_NAME}",    "is_sdxl": true  },  "datasets": [    {      "folder_path": "{DATASET_DIR}",      "caption_ext": "txt",      "mask_path": null,      "default_caption": "{TRIGGER_WORD} woman portrait, detailed face, high quality",      "resolution": {RESOLUTION},      "repeats": 10,      "crop_jitter": 20,      "crop_style": "center",      "crop_aspect": "square"    }  ],  "train": {    "batch_size": {BATCH_SIZE},    "steps": {NUM_EPOCHS * 150},    "lr": {LEARNING_RATE},    "noise_scheduler": "ddpm",    "train_unet": true,    "train_text_encoder": false,    "adam_weight_decay": 0.01,    "adam_beta1": 0.9,    "adam_beta2": 0.999,    "adam_epsilon": 1e-8,    "gradient_checkpointing": true,    "mixed_precision": "fp16"  },  "save": {    "dtype": "float16",    "save_every": 500,    "max_step_saves_to_keep": 5,    "save_dir": "{OUTPUT_DIR}"  },  "lora": {    "rank": {RANK},    "alpha": {ALPHA},    "dropout": 0.0  },  "sample": {    "sample_every": 250,    "prompts": [      "{TRIGGER_WORD} woman portrait, professional photo, studio lighting",      "{TRIGGER_WORD} woman, outdoor, natural lighting, detailed face",      "{TRIGGER_WORD} woman, close-up portrait, soft lighting"    ],    "negative_prompt": "blurry, low quality, distorted, deformed",    "seed": 42,    "walk_seed": true  }}'''config_path = '/content/alice_lora_config.json'with open(config_path, 'w') as f:    f.write(config)print(f'✅ Конфиг сохранён: {config_path}')print(f'   Trigger: {TRIGGER_WORD}')print(f'   Epochs: {NUM_EPOCHS}, Steps: {NUM_EPOCHS * 150}')print(f'   Rank: {RANK}, Alpha: {ALPHA}')print(f'   LR: {LEARNING_RATE}, Resolution: {RESOLUTION}')

## Шаг 3: Запуск тренировки 🚀

In [ ]:
#@title 🏋️ Тренировка LoRA%cd /content/ai-toolkitprint('🚀 Запуск тренировки...')print('Это займёт ~30-60 минут на T4 GPU')print('')!python run.py /content/alice_lora_config.jsonprint('')print('✅ Тренировка завершена!')print(f'LoRA сохранена в: {OUTPUT_DIR}')

## Шаг 4: Генерация тестовых изображений 🎨

In [ ]:
#@title 🖼️ Тестирование LoRAimport glob# Находим последний сохранённый LoRAlora_files = glob.glob(f'{OUTPUT_DIR}/**/*.safetensors', recursive=True)if not lora_files:    print('❌ LoRA файлы не найдены. Проверьте OUTPUT_DIR.')else:    latest_lora = max(lora_files, key=os.path.getmtime)    print(f'📦 LoRA: {latest_lora}')        # Устанавливаем diffusers для генерации    !pip install -q diffusers transformers accelerate        from diffusers import StableDiffusionXLPipeline    import torch        # Загружаем базовую модель    pipe = StableDiffusionXLPipeline.from_pretrained(        "stabilityai/stable-diffusion-xl-base-1.0",        torch_dtype=torch.float16,        use_safetensors=True    ).to('cuda')        # Загружаем LoRA    pipe.load_lora_weights(latest_lora)        # Генерируем    prompts = [        "alice woman portrait, professional photo, studio lighting, 8k",        "alice woman, outdoor park, natural lighting, detailed face",        "alice woman, close-up portrait, soft bokeh, high quality"    ]        for i, prompt in enumerate(prompts):        image = pipe(prompt, num_inference_steps=30, guidance_scale=7.5).images[0]        save_path = f'{OUTPUT_DIR}/test_{i+1}.png'        image.save(save_path)        print(f'✅ Сохранено: {save_path}')        print('\n🎉 Готово! Скачайте файлы из Google Drive.')